# 

# 03_compare_SVM_to_BERT

In this notebook, we will train a SVM model on the reduced dataset used the BERT performances for our task. We will then be able to compare the SVM performances with BERT for our task. 


**Inputs**

The reduced dataset in csv format, constructed in the first notebook. 

**Outputs**

The SVM model and its evaluation. 

## 1. Library import 

In [15]:
import pandas as pd 
import joblib
import random 

from sklearn.preprocessing import StandardScaler
from sklearn import metrics

from scipy.sparse import hstack, vstack
from scipy.sparse import csr_matrix
from sklearn import svm

from sklearn.model_selection import permutation_test_score
from sklearn.metrics import accuracy_score
import numpy as np

## 2. Data import  

In [16]:
# Download train data 
df_gpt = pd.read_csv("metrics//data//df_gpt.csv")
display(df_gpt)
# Download test data 
df_test = pd.read_csv("metrics//data//df_test_gpt.csv")

,text,paragraph,categorie,nombre_mots,tokens,pos_tags,function_word_freq,type_token_ratio,avg_word_length,noun_ratio,verb_ratio,adj_ratio,adv_ratio,exclamation_freq,question_freq,comma_freq,avg_sentence_length
0,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...","“So old Crouch lost it all, just when he thoug...",1,121,"['“', 'So', 'old', 'Crouch', 'lost', 'it', 'al...","['PUNCT', 'ADV', 'ADJ', 'PROPN', 'VERB', 'PRON...",0.451389,0.645833,3.777778,0.138889,0.138889,0.069444,0.069444,0.0,0.000000,0.090278,28.800000
1,Harry Potter and the Half-Blood Prince - Joann...,"""A bit… or more,"" said Dumbledore. ""You heard ...",1,100,"['""', 'A', 'bit', '…', 'or', 'more', ',', '""',...","['PUNCT', 'DET', 'NOUN', 'PUNCT', 'CCONJ', 'AD...",0.495726,0.615385,3.726496,0.094017,0.136752,0.068376,0.076923,0.0,0.000000,0.068376,29.250000
2,Burning Red - NoNameWriter.txt,And the culprit was decent to answer him prope...,0,134,"['And', 'the', 'culprit', 'was', 'decent', 'to...","['CCONJ', 'DET', 'NOUN', 'AUX', 'ADJ', 'PART',...",0.513158,0.671053,3.875000,0.171053,0.118421,0.078947,0.026316,0.0,0.000000,0.032895,19.000000
3,Burning Red - NoNameWriter.txt,"“Preventing damage, essentially. Petrification...",0,119,"['“', 'Preventing', 'damage', ',', 'essentiall...","['PUNCT', 'VERB', 'NOUN', 'PUNCT', 'ADV', 'PUN...",0.518248,0.686131,4.299270,0.175182,0.109489,0.087591,0.058394,0.0,0.000000,0.051095,27.400000
4,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...",The start of December brought wind and sleet t...,1,122,"['The', 'start', 'of', 'December', 'brought', ...","['DET', 'NOUN', 'ADP', 'PROPN', 'VERB', 'NOUN'...",0.446043,0.697842,4.338129,0.187050,0.093525,0.093525,0.035971,0.0,0.000000,0.043165,27.800000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,Burning Red - NoNameWriter.txt,Harry already knew using it in Diagon was okay...,0,133,"['Harry', 'already', 'knew', 'using', 'it', 'i...","['PROPN', 'ADV', 'VERB', 'VERB', 'PRON', 'ADP'...",0.559441,0.608392,4.384615,0.153846,0.132867,0.062937,0.062937,0.0,0.000000,0.034965,71.500000
396,Lily's Boy - SomewheresSword.txt,"“Lube?” Draco reminded, and Harry rolled his e...",0,116,"['“', 'Lube', '?', '”', 'Draco', 'reminded', '...","['PUNCT', 'NOUN', 'PUNCT', 'PUNCT', 'PROPN', '...",0.451852,0.607407,3.822222,0.185185,0.140741,0.044444,0.037037,0.0,0.007407,0.059259,27.000000
397,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...",What wouldn’t he have given to be one of these...,1,113,"['What', 'would', 'n’t', 'he', 'have', 'given'...","['PRON', 'AUX', 'PART', 'PRON', 'AUX', 'VERB',...",0.617188,0.632812,3.937500,0.078125,0.125000,0.031250,0.046875,0.0,0.007812,0.039062,21.333333
398,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...","He felt much calmer, somehow, now that he was ...",1,176,"['He', 'felt', 'much', 'calmer', ',', 'somehow...","['PRON', 'VERB', 'ADV', 'ADJ', 'PUNCT', 'ADV',...",0.482587,0.572139,3.950249,0.154229,0.129353,0.049751,0.049751,0.0,0.000000,0.064677,25.125000


In [17]:
# Load tf-idf vectorizer
tfidf = joblib.load("metrics/data/tfidf_vectorizer.pkl")

## 3. Split train test 

### 3.1 Training columns 

In [18]:

stylometric_columns = [
    "function_word_freq",
    "type_token_ratio",
    "avg_word_length",
    "noun_ratio",
    "verb_ratio",
    "adj_ratio",
    "adv_ratio",
    "exclamation_freq",
    "question_freq",
    "comma_freq",
    "avg_sentence_length"
]



### 3.2 X_train and y_train 

In [19]:
# Construct the tf-idf matrix
X_tfidf_gpt = tfidf.transform(df_gpt["paragraph"])
X_tfidf = tfidf.transform(df_test["paragraph"])

In [20]:

# We first create X_train and y_train by using df_gpt
X_stylo = df_gpt[stylometric_columns].values


X_train = hstack([
    X_tfidf_gpt,
    csr_matrix(X_stylo)
])

y_train = df_gpt["categorie"]


### 3.3 X_test and y_test 

In [21]:

# Stylometric features
X_stylo = df_test[stylometric_columns].values

# TF-IDF features
X_tfidf = tfidf.transform(df_test["paragraph"])

# Combined feature matrix
X_test = hstack([
    X_tfidf,
    csr_matrix(X_stylo)
])

# Labels
y_test = df_test["categorie"]


### 3.4 Normalize data 

In [22]:
# We normalize the data 
scaler = StandardScaler(with_mean=False)  

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 4. Data modeling 

### 4.1 Naive model 

In [23]:
# We compute a naive model by predicting a random categorie each time 
y_pred_naive = [random.choice([0, 1])]*len(y_test)

print("---Performances of the Naive Model---\n")
print("Accuracy:" ,metrics.accuracy_score(y_pred_naive, y_test))

---Performances of the Naive Model---

Accuracy: 0.5


### 4.2 SMV model 

### 4.2.1 Training 

In [24]:

#Create a svm Classifier
model_svm = svm.SVC(kernel='poly', class_weight="balanced") # Polynomial Kernel

#Train the model using the training sets
model_svm.fit(X_train, y_train)

#Predict the response for test dataset
y_pred_svm_train = model_svm.predict(X_train)
y_pred_svm = model_svm.predict(X_test)

### 4.2.2 Evaluation 

In [29]:


# Combine train + test (IMPORTANT pour le test de permutation)
X = vstack([X_train, X_test])
y = np.concatenate([y_train, y_test])

# Define the model again (same configuration)
model = svm.SVC(kernel='poly', class_weight="balanced")

# Compute permutation test
score, permutation_scores, p_value = permutation_test_score(
    model,
    X,
    y,
    scoring="accuracy",
    cv=5,              # cross-validation
    n_permutations=100,  # increase to 1000 for final report
    n_jobs=-1
)

print("Model accuracy:", score)
print("F1-Score:",metrics.f1_score(y_pred_svm, y_test))
print("Permutation p-value:", p_value)

Model accuracy: 0.812
F1-Score: 0.8114285714285714
Permutation p-value: 0.009900990099009901


## 5. Models Download 

In [27]:
def download_model(model, results_dir: str, filename: str = "model.pkl"):
    """
    Save a trained model to disk using joblib.
    """
    
    model_path = f"{results_dir}//{filename}"
    joblib.dump(model, model_path)

    print(f"✅ Model saved to {model_path}")


In [28]:
download_model(model_svm, "metrics//models", "final_svm.pkl")

✅ Model saved to metrics//models//final_svm.pkl
